# Python Descriptors: An Advanced, Step-by-Step Problem-Solving Tutorial

**One notebook · 15 new problems · fully worked solutions · standard-library-only Python**

This notebook follows the style of a live lesson. Each problem begins with a requirement, establishes the relevant rule, introduces code in small increments, explains the result, and finishes with a solution and tests. The problems use new applications rather than reproducing the card-deck examples in the source lesson.

**How to study:** Run cells from top to bottom. Before revealing each solution, try the proposed design yourself. Read the explanations after running the preceding cell: many of the subtleties concern *which object* Python passes into `__get__` and *which storage location* an assignment changes.

**Environment:** Python 3.10+ recommended. Everything uses the Python standard library. Tests use ordinary `assert` statements so the notebook is runnable in JupyterLab, classic Jupyter, and Colab.

## Road map

| Block | Problems | Key ideas |
| --- | --- | --- |
| Lookup mechanics | 1–2 | owner/instance access, class inspection, data versus non-data precedence |
| Reusable fields | 3–4 | `__set_name__`, validation, write-once rules |
| State and caching | 5–8 | isolated RNG, lazy values, TTL expiry, weak-key storage |
| Behavior descriptors | 9–11 | method binding, observers, derived and writable measurements |
| Larger systems | 12–15 | inherited field registry, static inspection, atomic invariants, `__slots__` |

**Important correction to a common simplification:** a descriptor is a *data* descriptor if its type defines `__set__` **or** `__delete__`. Non-data descriptors implement `__get__` without either of those methods. The descriptor usually belongs to the class; per-object state generally should not live directly on a shared descriptor object.

## A compact lookup model (refer back to it)

For an ordinary instance attribute lookup `obj.name`, think of the following simplified order:

1. Look up `name` in the class and its MRO. If the result is a **data descriptor**, invoke it.
2. Otherwise check `obj.__dict__`, if one exists.
3. Otherwise invoke a **non-data descriptor** from the class, if found.
4. Otherwise return the ordinary class attribute; if nothing exists, raise `AttributeError`.

The actual machinery is implemented by `object.__getattribute__`, with special cases for class access, `super()`, and custom attribute-access methods. Descriptor methods are normally activated when a descriptor object is stored as a **class attribute**, not merely when placed in an instance dictionary.

# Problem 01 — A descriptor that explains class access

**Challenge.** Expose a reproducible timestamp while keeping the descriptor itself inspectable from the owning class.

**By the end, you should be able to:** implement `__get__` correctly, inject a clock, and inspect the raw class dictionary.

We will move from a small experiment to a complete implementation, then check the behavior with executable assertions.

### Step 1 — Identify the two lookup paths

When `event.time` is evaluated, `instance` is the `event` object. When `Event.time` is evaluated, `instance` is `None` and `owner` is `Event`. Returning the descriptor on class access makes documentation, introspection, and configuration easier.

We inject a clock rather than relying on the wall clock. That makes our example deterministic and easy to test.

In [1]:
from datetime import datetime, timezone

class UTCStamp:
    def __init__(self, clock):
        self.clock = clock

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return self.clock().isoformat()

fixed_clock = lambda: datetime(2030, 1, 2, 3, 4, 5, tzinfo=timezone.utc)

class Event:
    time = UTCStamp(fixed_clock)

event = Event()
print('instance:', event.time)
print('class exposes descriptor:', Event.time is Event.__dict__['time'])
assert event.time == '2030-01-02T03:04:05+00:00'
assert Event.time is Event.__dict__['time']

instance: 2030-01-02T03:04:05+00:00
class exposes descriptor: True


### Step 2 — Explain the result

`Event.__dict__['time']` retrieves the descriptor object without ordinary dotted-attribute lookup. `Event.time` calls the descriptor with `instance=None`; our method returns `self`. `event.time` requests a newly formatted timestamp.

Unlike an old `datetime.utcnow()` example, this uses a timezone-aware UTC value. Note that `UTCStamp` is **non-data**: it implements `__get__` but not `__set__` or `__delete__`. We will exploit that in the next problem.

### Complete solution and verification

The complete implementation is already above. These final checks emphasize object identity, not incidental output formatting. We also demonstrate how an instance can shadow a non-data descriptor.

In [2]:
event.time = 'manually overridden'
assert event.time == 'manually overridden'
assert Event.time is Event.__dict__['time']
del event.time
assert event.time == '2030-01-02T03:04:05+00:00'
print('PASS: class access, UTC value, shadowing and unshadowing')

PASS: class access, UTC value, shadowing and unshadowing


**Takeaway.** A descriptor should generally return itself when accessed through its class. In contrast, instance assignment can hide a non-data descriptor if the instance has a dictionary. This behavior is a language rule, not an accident.

# Problem 02 — Predict the exact attribute-precedence order

**Challenge.** Create two descriptors that look similar but differ in whether an instance attribute can override them.

**By the end, you should be able to:** differentiate data and non-data descriptors, including a descriptor whose only mutator is `__delete__`.

We will move from a small experiment to a complete implementation, then check the behavior with executable assertions.

### Step 1 — Build probes instead of guessing

We want a clear experiment where `__get__` records its own execution. `ReadProbe` is non-data. `DataProbe` supplies `__set__`, making it a data descriptor. A third probe defines `__delete__` but no `__set__`: this is *also* a data descriptor.

No instance state is stored on the probe objects.

In [3]:
class ReadProbe:
    def __get__(self, instance, owner):
        return self if instance is None else 'non-data result'

class DataProbe:
    def __get__(self, instance, owner):
        return self if instance is None else 'data result'

    def __set__(self, instance, value):
        instance.__dict__['_data_value'] = value

class DeleteOnlyProbe:
    def __get__(self, instance, owner):
        return self if instance is None else 'delete-only result'

    def __delete__(self, instance):
        instance.__dict__.pop('_delete_only_internal', None)

class PrecedenceDemo:
    read = ReadProbe()
    data = DataProbe()
    delete_only = DeleteOnlyProbe()

p = PrecedenceDemo()
p.__dict__.update(read='instance read', data='instance data',
                  delete_only='instance delete')
print(p.read, '|', p.data, '|', p.delete_only)

instance read | data result | delete-only result


### Step 2 — Derive the answer from the model

The `read` descriptor loses to the instance dictionary. The `data` and `delete_only` descriptors win *even though the instance dictionary contains keys of exactly the same names*. Data-descriptor classification depends on methods defined by the **descriptor's type**, not on whether an assignment was successfully performed.

In [4]:
assert p.read == 'instance read'
assert p.data == 'data result'
assert p.delete_only == 'delete-only result'
assert type(PrecedenceDemo.__dict__['delete_only']).__delete__ is DeleteOnlyProbe.__delete__

p.data = 12
assert p.__dict__['_data_value'] == 12
assert p.data == 'data result'  # __get__ is still authoritative
print('PASS: data descriptor takes priority over instance dictionary')

PASS: data descriptor takes priority over instance dictionary


### Step 3 — Predict deletion carefully

`del p.read` removes the *instance* entry because the non-data descriptor supplies no deletion hook. `del p.delete_only` invokes the descriptor's `__delete__` hook even though a same-named instance entry exists. Our deliberately minimal deletion hook leaves that same-named entry untouched: descriptor lookup still wins afterward.

In production, `__get__` and `__delete__` should agree on a real backing store; this example separates mechanics from application design.

In [5]:
del p.read
assert p.read == 'non-data result'
del p.delete_only
assert p.__dict__['delete_only'] == 'instance delete'
assert p.delete_only == 'delete-only result'
print('PASS: deletion follows data versus non-data precedence')

PASS: deletion follows data versus non-data precedence


**Takeaway.** `__get__` alone means non-data; either `__set__` or `__delete__` makes a data descriptor. Do not conclude that a descriptor is non-data merely because it rejects writes.

# Problem 03 — Build independently named, strictly validated fields

**Challenge.** Define several numeric fields using one reusable descriptor class without sharing their values.

**By the end, you should be able to:** capture field names with `__set_name__`, select private backing keys, reject booleans in numeric input, and raise contextual errors.

We will move from a small experiment to a complete implementation, then check the behavior with executable assertions.

### Step 1 — State the storage invariant

A descriptor instance stored on `Invoice.tax_rate` is shared by *all* `Invoice` objects. Storing the last assigned tax rate on the descriptor would therefore leak values between invoices. Instead, put each value in the owning instance's `__dict__` under a private key constructed by `__set_name__`.

Python calls `__set_name__(owner, name)` during class creation for descriptors defined in that class body. The hook is not called again just because a subclass inherits the descriptor.

In [6]:
import math

class BoundedNumber:
    def __init__(self, *, minimum, maximum):
        if not minimum <= maximum:
            raise ValueError('minimum must not exceed maximum')
        self.minimum = minimum
        self.maximum = maximum

    def __set_name__(self, owner, name):
        self.name = name
        self.storage_key = f'_{name}'

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.storage_key]
        except KeyError:
            raise AttributeError(f'{self.name} has not been initialized') from None

    def __set__(self, instance, value):
        if (isinstance(value, bool) or not isinstance(value, (int, float))
                or not math.isfinite(value)):
            raise TypeError(f'{self.name} must be a finite real number (not bool)')
        if not self.minimum <= value <= self.maximum:
            raise ValueError(f'{self.name} must be between {self.minimum} and {self.maximum}')
        instance.__dict__[self.storage_key] = value

### Step 2 — Install the same descriptor **class**, but not the same descriptor **object**, twice

Every declaration below creates a fresh `BoundedNumber` object, so each has its own metadata. We avoid using a single shared descriptor for unrelated names, because its `__set_name__` metadata would be overwritten by the second declaration.

In [7]:
class Invoice:
    tax_rate = BoundedNumber(minimum=0, maximum=100)
    discount = BoundedNumber(minimum=0, maximum=100)

    def __init__(self, tax_rate, discount):
        self.tax_rate = tax_rate
        self.discount = discount

first = Invoice(7.5, 10)
second = Invoice(19, 0)
print(first.__dict__)
print(second.__dict__)
assert (first.tax_rate, first.discount) == (7.5, 10)
assert (second.tax_rate, second.discount) == (19, 0)
assert Invoice.tax_rate.name == 'tax_rate'
assert Invoice.discount.name == 'discount'

{'_tax_rate': 7.5, '_discount': 10}
{'_tax_rate': 19, '_discount': 0}


### Step 3 — Test the uncomfortable inputs

In Python, `isinstance(True, int)` is true. When the business rule requires an actual number but not a Boolean, check for `bool` explicitly. Also exclude infinity and NaN; simple range comparisons alone can behave unexpectedly for NaN.

In [8]:
def expect_error(error_type, action):
    try:
        action()
    except error_type:
        return
    except Exception as exc:
        raise AssertionError(f'Expected {error_type.__name__}, got {type(exc).__name__}') from exc
    raise AssertionError(f'Expected {error_type.__name__}, but no exception was raised')

for invalid in (True, float('nan'), float('inf'), '10'):
    expect_error(TypeError, lambda invalid=invalid: setattr(first, 'tax_rate', invalid))
expect_error(ValueError, lambda: setattr(first, 'discount', 101))
assert first.tax_rate == 7.5
assert first.discount == 10
print('PASS: independent fields, meaningful failures, state unchanged')

PASS: independent fields, meaningful failures, state unchanged


### Complete solution: an uninitialized field

A descriptor need not invent a default value. If the owner did not assign the field, `AttributeError` is idiomatic and plays well with ordinary Python attribute semantics. Notice that we translate only the expected `KeyError`, not every exception.

In [9]:
uninitialized = Invoice.__new__(Invoice)
expect_error(AttributeError, lambda: uninitialized.tax_rate)
assert '_tax_rate' not in uninitialized.__dict__
print('PASS: missing value handled explicitly')

PASS: missing value handled explicitly


**Takeaway.** Use `__set_name__` for field-specific metadata, store per-instance values on the instance, and validate *before* mutation. Reusable descriptor classes are usually more useful than sharing one descriptor instance across unrelated fields.

# Problem 04 — Implement a write-once configuration value

**Challenge.** Permit exactly one assignment per owner instance while allowing a legitimate `None` value.

**By the end, you should be able to:** use a sentinel and a data descriptor, distinguish missing from `None`, and prevent ordinary reassignment or deletion.

We will move from a small experiment to a complete implementation, then check the behavior with executable assertions.

### Step 1 — Decide what “unset” means

A field containing `None` is still set. Thus `dict.get(key) is None` is not an acceptable initialization check. Membership testing against the backing dictionary is sufficient here.

A write-once rule is an ordinary API invariant, **not a security boundary**: arbitrary Python code that edits `obj.__dict__` can deliberately bypass this simple implementation.

In [10]:
class WriteOnce:
    def __set_name__(self, owner, name):
        self.name = name
        self.key = f'_once_{name}'

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if self.key not in instance.__dict__:
            raise AttributeError(f'{self.name} has not been set')
        return instance.__dict__[self.key]

    def __set__(self, instance, value):
        if self.key in instance.__dict__:
            raise AttributeError(f'{self.name} can only be assigned once')
        instance.__dict__[self.key] = value

    def __delete__(self, instance):
        raise AttributeError(f'{self.name} cannot be deleted')

### Step 2 — Use the descriptor in an ordinary constructor

Because the value is stored on the instance, independent configuration objects can receive independent values. The descriptor does not retain those values or create a global singleton.

In [11]:
class JobConfig:
    job_id = WriteOnce()
    optional_label = WriteOnce()

    def __init__(self, job_id, optional_label=None):
        self.job_id = job_id
        self.optional_label = optional_label

job_a = JobConfig('A-001')
job_b = JobConfig('B-002', 'batch')
assert job_a.optional_label is None
assert job_b.job_id == 'B-002'
expect_error(AttributeError, lambda: setattr(job_a, 'job_id', 'changed'))
expect_error(AttributeError, lambda: setattr(job_a, 'optional_label', 'changed'))
expect_error(AttributeError, lambda: delattr(job_a, 'job_id'))
print('PASS: None counts as an assigned value; write and delete blocked')

PASS: None counts as an assigned value; write and delete blocked


### Complete solution and edge case

With a data descriptor, even inserting a same-named key such as `job_a.__dict__['job_id']` does not shadow normal lookup. The descriptor's storage key remains authoritative. Again, direct mutation of the *private* backing key is possible by hostile or careless code, so do not call this tamper-proof.

In [12]:
job_a.__dict__['job_id'] = 'misleading shadow'
assert job_a.job_id == 'A-001'
expect_error(AttributeError, lambda: JobConfig.__new__(JobConfig).job_id)
print('PASS: descriptor precedence and missing-state behavior')

PASS: descriptor precedence and missing-state behavior


**Takeaway.** Data descriptors are useful for enforcing conventional API contracts. Membership checks distinguish a missing field from a field holding a falsey value.

# Problem 05 — Create reproducible random choices without global state

**Challenge.** Build a random-choice descriptor where different owners do not accidentally share a random generator.

**By the end, you should be able to:** inject a generator factory, create per-instance random state, and avoid changing `random.seed()` globally.

We will move from a small experiment to a complete implementation, then check the behavior with executable assertions.

### Step 1 — Think about the stateful dependency

A global random generator couples unrelated tests and objects. A single `Random` instance stored on the descriptor would also be shared between every owner instance. Instead, give each owner a generator stored under a field-specific private key. Each generator can be seeded for reproducible examples.

The descriptor itself remains non-data: assignment to the public name can intentionally override it.

In [13]:
import random

class RandomChoice:
    def __init__(self, *options, rng_factory=None):
        if not options:
            raise ValueError('at least one option is required')
        self.options = tuple(options)
        self.rng_factory = rng_factory or random.Random

    def __set_name__(self, owner, name):
        self.name = name
        self.key = f'_rng_for_{name}'

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if self.key not in instance.__dict__:
            instance.__dict__[self.key] = self.rng_factory()
        return instance.__dict__[self.key].choice(self.options)

### Step 2 — Provide reproducible seeds and verify isolation

The factory `lambda: random.Random(17)` makes *new* generators with the same initial state; it does not share one generator. Two instances therefore produce the same seeded sequence when read equally often, while reads from one instance do not advance the other.

In [14]:
class Simulator:
    outcome = RandomChoice('success', 'retry', 'fail',
                           rng_factory=lambda: random.Random(17))
    priority = RandomChoice(1, 2, 3, rng_factory=lambda: random.Random(99))

sim_a = Simulator()
sim_b = Simulator()
sequence_a = [sim_a.outcome for _ in range(8)]
sequence_b = [sim_b.outcome for _ in range(8)]
assert sequence_a == sequence_b
assert sim_a.__dict__['_rng_for_outcome'] is not sim_b.__dict__['_rng_for_outcome']
assert set(sequence_a) <= {'success', 'retry', 'fail'}
print('reproducible sequence:', sequence_a)

reproducible sequence: ['fail', 'retry', 'retry', 'retry', 'retry', 'success', 'fail', 'fail']


### Step 3 — Understand what is and is not independent

Two different field names receive two distinct stored generators. Two instances receive separate state. A user-provided factory *could* deliberately return the same generator each time; honoring that injection is the caller's responsibility.

This describes sampling **with replacement**: reading an attribute can return the same option repeatedly. It is not a deck that removes cards.

In [15]:
assert sim_a.__dict__['_rng_for_outcome'] is not sim_a.__dict__.get('_rng_for_priority')
_ = sim_a.priority
assert sim_a.__dict__['_rng_for_outcome'] is not sim_a.__dict__['_rng_for_priority']
expect_error(ValueError, lambda: RandomChoice())
assert Simulator.outcome.options == ('success', 'retry', 'fail')
print('PASS: input validation, per-field RNG, introspectable class access')

PASS: input validation, per-field RNG, introspectable class access


**Takeaway.** Dependency injection and instance-local state make random behavior testable. Keep the descriptor's configuration on the descriptor, and the mutable generator on the owner instance.

# Problem 06 — Build a lazy non-data descriptor with deliberate shadowing

**Challenge.** Calculate an expensive value on first access and then bypass the descriptor on subsequent accesses.

**By the end, you should be able to:** write a small `cached_property`-like descriptor and invalidate it through normal deletion.

We will move from a small experiment to a complete implementation, then check the behavior with executable assertions.

### Step 1 — Exploit precedence instead of fighting it

A non-data descriptor is invoked only until an instance dictionary entry with the same name exists. On the first read, compute the value and insert it into `instance.__dict__[name]`. On later reads, the instance dictionary wins and `__get__` is not called.

This design deliberately requires an instance with `__dict__`. Slotted-only instances need another storage strategy, which we will explore later.

In [16]:
class LazyValue:
    def __init__(self, function):
        self.function = function
        self.__doc__ = function.__doc__

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        result = self.function(instance)
        instance.__dict__[self.name] = result
        return result

### Step 2 — Count expensive calculations, not just return values

The example uses a plain counter instead of a real network call, which keeps tests fast and deterministic. A descriptor should not cache a failure unless that behavior is explicitly part of its contract: this implementation stores only after a successful computation.

In [17]:
class AnalyticsReport:
    def __init__(self, observations):
        self.observations = tuple(observations)
        self.calculations = 0

    @LazyValue
    def average(self):
        "Calculate once, until the cached attribute is removed."
        self.calculations += 1
        return sum(self.observations) / len(self.observations)

report = AnalyticsReport([10, 20, 30])
assert report.average == 20
assert report.average == 20
assert report.calculations == 1
assert report.__dict__['average'] == 20
assert AnalyticsReport.average.name == 'average'
print('PASS: expensive function executed once')

PASS: expensive function executed once


### Step 3 — Invalidate explicitly

Deletion of `report.average` deletes the cached instance key; the next lookup reaches the descriptor again. If `observations` changes, the application must invalidate the cache. Without an explicit dependency mechanism, there is no automatic relationship between the two attributes.

In [18]:
report.observations = (10, 50)
del report.average
assert report.average == 30
assert report.calculations == 2
report.average = -999  # non-data descriptor can be shadowed deliberately
assert report.average == -999
del report.average
assert report.average == 30
assert report.calculations == 3
print('PASS: deletion recomputes; assignment can shadow non-data descriptor')

PASS: deletion recomputes; assignment can shadow non-data descriptor


### Complete solution: failed computation is not cached

A bad input leads to a real exception. Since the store operation follows the function call, there is no `average` key afterward. Fixing the input and retrying works.

In [19]:
empty_report = AnalyticsReport([])
expect_error(ZeroDivisionError, lambda: empty_report.average)
assert 'average' not in empty_report.__dict__
empty_report.observations = (8, 12)
assert empty_report.average == 10
print('PASS: errors do not poison the cache')

PASS: errors do not poison the cache


**Takeaway.** A lazy cache can be exceptionally short precisely because it is **non-data**. The tradeoff is explicit invalidation, the need for an instance dictionary, and potential duplicate work under concurrent first reads.

# Problem 07 — Extend caching with a time-to-live and deterministic clock

**Challenge.** Cache a value for a fixed duration, provide explicit invalidation, and reject direct assignment.

**By the end, you should be able to:** choose data-descriptor behavior, use monotonic time, and test expiry without sleeping.

We will move from a small experiment to a complete implementation, then check the behavior with executable assertions.

### Step 1 — Write the intended contract

The value is computed on the first access, reused while `now < expires_at`, and recomputed at or after the boundary. `del obj.value` invalidates the stored cache. Direct assignment is rejected.

Use an injected *monotonic* clock rather than local calendar time: elapsed durations should not depend on timezone changes or wall-clock corrections. Here a `FakeClock` lets us advance time without `sleep`.

In [20]:
import time

class FakeClock:
    def __init__(self, initial=0.0):
        self.current = float(initial)

    def __call__(self):
        return self.current

    def advance(self, seconds):
        self.current += seconds

class TTLValue:
    def __init__(self, function, *, seconds, clock=time.monotonic):
        if isinstance(seconds, bool) or not isinstance(seconds, (int, float)):
            raise TypeError('seconds must be numeric')
        if not math.isfinite(seconds) or seconds <= 0:
            raise ValueError('seconds must be positive and finite')
        self.function = function
        self.seconds = seconds
        self.clock = clock

    def __set_name__(self, owner, name):
        self.name = name
        self.key = f'_ttl_cache_{name}'

    def __get__(self, instance, owner):
        if instance is None:
            return self
        now = self.clock()
        previous = instance.__dict__.get(self.key)
        if previous is not None:
            expiry, value = previous
            if now < expiry:
                return value
        value = self.function(instance)
        instance.__dict__[self.key] = (self.clock() + self.seconds, value)
        return value

    def __set__(self, instance, value):
        raise AttributeError(f'{self.name} is read-only; delete it to invalidate')

    def __delete__(self, instance):
        instance.__dict__.pop(self.key, None)

### Step 2 — Connect a function to a descriptor

A decorator would be possible, but direct construction makes the clock and TTL visible in this exercise. We put the fake clock in the class declaration so all reads consult the same controllable timeline. Cache entries remain separate in each instance's dictionary.

In [21]:
ttl_clock = FakeClock()

def load_snapshot(instance):
    instance.loads += 1
    return (instance.identity, instance.loads)

class SnapshotClient:
    snapshot = TTLValue(load_snapshot, seconds=10, clock=ttl_clock)

    def __init__(self, identity):
        self.identity = identity
        self.loads = 0

client_a = SnapshotClient('A')
client_b = SnapshotClient('B')
assert client_a.snapshot == ('A', 1)
assert client_a.snapshot == ('A', 1)
assert client_b.snapshot == ('B', 1)
assert (client_a.loads, client_b.loads) == (1, 1)
print('PASS: first read computes once per instance')

PASS: first read computes once per instance


### Step 3 — Test the *boundary*, not only the happy path

The cache remains fresh just before ten seconds. At exactly ten seconds, the contract says it expires. Invalidation also works before expiry. The second clock read, taken after computation, sets the expiration from completion time; for expensive functions this avoids counting the computation itself as cache lifetime.

In [22]:
ttl_clock.advance(9.999)
assert client_a.snapshot == ('A', 1)
ttl_clock.advance(0.001)
assert client_a.snapshot == ('A', 2)
assert client_b.snapshot == ('B', 2)
del client_a.snapshot
assert client_a.snapshot == ('A', 3)
expect_error(AttributeError, lambda: setattr(client_a, 'snapshot', 'override'))
assert SnapshotClient.snapshot.seconds == 10
print('PASS: exact boundary, per-instance expiry, manual invalidation')

PASS: exact boundary, per-instance expiry, manual invalidation


### Complete solution: check failure handling

Because the code stores the cache only after computing successfully, a raised exception leaves the old expired entry in place. On the next read it is still expired and the computation is retried. This implementation is synchronous and does not prevent duplicate work when several threads read an expired value concurrently; adding synchronization is a separate requirement.

In [23]:
failure_clock = FakeClock()

class Flaky:
    def __init__(self):
        self.calls = 0

    def compute(self):
        self.calls += 1
        if self.calls == 1:
            raise RuntimeError('temporary failure')
        return 42

    value = TTLValue(compute, seconds=5, clock=failure_clock)

flaky = Flaky()
expect_error(RuntimeError, lambda: flaky.value)
assert '_ttl_cache_value' not in flaky.__dict__
assert flaky.value == 42
assert flaky.calls == 2
print('PASS: failed computation is retried')

PASS: failed computation is retried


**Takeaway.** Time-based caching needs an explicit expiry rule, a monotonic clock, a way to invalidate, and a clear failure policy. A data descriptor prevents public assignment from silently replacing the caching behavior.

# Problem 08 — Store values on slotted owners with weak references

**Challenge.** Make a descriptor work when its owner has no `__dict__`, without keeping owner instances alive forever.

**By the end, you should be able to:** use `WeakKeyDictionary` and explain the weak-reference and hashability requirements.

We will move from a small experiment to a complete implementation, then check the behavior with executable assertions.

### Step 1 — Why the earlier storage pattern fails

A class declaring `__slots__` without `__dict__` has no normal per-instance dictionary. A descriptor that assumes `instance.__dict__` will fail. One alternative is a separate backing slot (Problem 15). Another is an external weak-key mapping owned by the descriptor.

`weakref.WeakKeyDictionary` does not retain the owner instance through its key. It does, however, require instances that support weak references and are hashable. Avoid storing the owner itself in the mapping's values, or you would accidentally retain it.

In [24]:
import weakref
import gc

class WeakField:
    def __init__(self, *, validator=lambda value: True):
        self.values = weakref.WeakKeyDictionary()
        self.validator = validator

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return self.values[instance]
        except KeyError:
            raise AttributeError(f'{self.name} has not been assigned') from None

    def __set__(self, instance, value):
        if not self.validator(value):
            raise ValueError(f'invalid {self.name}: {value!r}')
        self.values[instance] = value

    def __delete__(self, instance):
        try:
            del self.values[instance]
        except KeyError:
            raise AttributeError(f'{self.name} is missing') from None

### Step 2 — Make a slotted, weak-referenceable owner

The `__weakref__` slot permits weak references. This class inherits default identity equality and hashing, which is appropriate for weak-key storage here. A custom equality/hash method based on mutable data would complicate key identity and is not appropriate without careful design.

In [25]:
class Device:
    __slots__ = ('__weakref__',)
    temperature = WeakField(validator=lambda value: type(value) in (int, float))

x = Device()
y = Device()
assert not hasattr(x, '__dict__')
x.temperature = 18
y.temperature = 23
assert (x.temperature, y.temperature) == (18, 23)
expect_error(ValueError, lambda: setattr(x, 'temperature', 'hot'))
assert x.temperature == 18
print('PASS: no __dict__ needed; values remain independent')

PASS: no __dict__ needed; values remain independent


### Step 3 — Verify that owners can disappear

The descriptor owns the weak-key dictionary, but it should not prevent garbage collection of otherwise unreferenced owners. We count entries after removing the only strong reference to a temporary device. `gc.collect()` is used only in this deliberately small lifecycle test, not as an application-level cleanup strategy.

In [26]:
field = Device.temperature
count_before = len(field.values)
temporary = Device()
temporary.temperature = 99
reference = weakref.ref(temporary)
assert len(field.values) == count_before + 1
del temporary
gc.collect()
assert reference() is None
assert len(field.values) == count_before
print('PASS: weak external storage does not keep owners alive')

PASS: weak external storage does not keep owners alive


### Complete solution: deletion and a limitation

Deletion affects only the targeted owner. Weak-key storage is unsuitable for owners that cannot be weak-referenced, and its keys need a sound hash/equality contract. Values should not hold strong references back to the owners. For most ordinary classes, simpler `instance.__dict__` storage remains the preferred option.

In [27]:
del x.temperature
expect_error(AttributeError, lambda: x.temperature)
assert y.temperature == 23
assert Device.temperature is field
print('PASS: deletion is per instance')

PASS: deletion is per instance


**Takeaway.** External weak-key state expands descriptor compatibility, but comes with real object-model constraints. Select a storage strategy consciously rather than assuming every owner has `__dict__`.

# Problem 09 — Reconstruct method binding and add an access rule

**Challenge.** Understand why ordinary functions become bound methods and implement a guarded method as a descriptor.

**By the end, you should be able to:** return a callable bound to the instance and preserve method metadata with `functools.wraps`.

We will move from a small experiment to a complete implementation, then check the behavior with executable assertions.

### Step 1 — See the native model

Functions are non-data descriptors: retrieving a function from an instance normally produces a bound method. A bound method remembers the instance in `__self__` and the underlying function in `__func__`.

Our custom descriptor will wrap a function and check an instance permission before the function runs. Returning the descriptor on class access keeps it inspectable; this is an intentionally different class-access API from ordinary functions.

In [28]:
from types import MethodType
from functools import wraps

class NativeMethodExample:
    def double(self, number):
        return 2 * number

native = NativeMethodExample()
assert native.double.__self__ is native
assert native.double.__func__ is NativeMethodExample.__dict__['double']
assert native.double(6) == 12
print('PASS: ordinary functions use the descriptor protocol')

PASS: ordinary functions use the descriptor protocol


### Step 2 — Construct a fresh wrapper upon access

`MethodType(wrapper, instance)` creates a bound method and passes the instance as the wrapper's first argument. `functools.wraps` preserves the original function's name, documentation, and `__wrapped__` link for common inspection tools.

In [29]:
class GuardedMethod:
    def __init__(self, function):
        self.function = function
        self.__doc__ = function.__doc__

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self

        @wraps(self.function)
        def guarded(target, *args, **kwargs):
            if not target.can_execute:
                raise PermissionError(f'{self.name} requires can_execute=True')
            return self.function(target, *args, **kwargs)

        return MethodType(guarded, instance)

class AdminAction:
    def __init__(self, can_execute):
        self.can_execute = can_execute

    @GuardedMethod
    def run(self, amount):
        "Perform one permitted operation."
        return amount * 3

### Step 3 — Verify both method binding and access control

The guard checks permission *at call time*, not only when the bound method is fetched. This matters when a method reference is retained while the object's permission changes. This is still application-level access control, not protection against malicious code that can access private members or the underlying function.

In [30]:
admin = AdminAction(True)
method = admin.run
assert method.__self__ is admin
assert method.__name__ == 'run'
assert method(4) == 12
admin.can_execute = False
expect_error(PermissionError, lambda: method(4))
assert isinstance(AdminAction.run, GuardedMethod)
assert AdminAction.run.function.__name__ == 'run'
print('PASS: binding and guard-on-call behavior')

PASS: binding and guard-on-call behavior


### Complete solution: a subtle override

`GuardedMethod` is a non-data descriptor. An instance with `__dict__` can replace `admin.run` with a new callable. If the application requires that public assignment not override the guard, define a `__set__` that raises. A restrictive API should also avoid exposing the raw protected operation through other public routes.

In [31]:
admin.run = lambda amount: 'overridden'
assert admin.run(10) == 'overridden'
del admin.run
expect_error(PermissionError, lambda: admin.run(10))
print('PASS: deliberate non-data descriptor shadowing documented')

PASS: deliberate non-data descriptor shadowing documented


**Takeaway.** Python's method binding is itself a descriptor application. For guarded behavior, decide explicitly whether shadowing is acceptable and whether checks belong at access time or call time.

# Problem 10 — Build an observable, validated model field

**Challenge.** Notify listeners only when a valid value actually changes, without notifying about failed updates.

**By the end, you should be able to:** validate first, commit the change, then publish the event, while documenting callback-failure semantics.

We will move from a small experiment to a complete implementation, then check the behavior with executable assertions.

### Step 1 — Choose observable semantics

We will emit events only for changes *after the initial assignment*. Invalid input must leave the old value and event log unchanged. Reassigning an equal value does nothing. The listener list belongs to each owner instance, not to the descriptor.

**Important tradeoff:** notification happens after the state is committed. If a listener raises, the exception propagates and the new state remains committed. Rolling back arbitrary listeners is not safe because external side effects may already have happened.

In [32]:
_MISSING = object()

class ObservedNumber:
    def __init__(self, *, minimum=0):
        self.minimum = minimum

    def __set_name__(self, owner, name):
        self.name = name
        self.key = f'_observed_{name}'

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if self.key not in instance.__dict__:
            raise AttributeError(f'{self.name} not initialized')
        return instance.__dict__[self.key]

    def __set__(self, instance, value):
        if (type(value) not in (int, float) or not math.isfinite(value)
                or value < self.minimum):
            raise ValueError(f'{self.name} requires a finite number >= {self.minimum}')
        old = instance.__dict__.get(self.key, _MISSING)
        if old is not _MISSING and old == value:
            return
        instance.__dict__[self.key] = value
        if old is not _MISSING:
            for listener in tuple(instance.__dict__.get('_listeners', ())):
                listener(self.name, old, value)

### Step 2 — Keep a per-instance subscription list

Subscribers are ordinary callables. Taking a tuple snapshot before notification gives predictable iteration if a callback adds or removes listeners during an event. This implementation is synchronous: listeners run in the setter's call stack.

In [33]:
class BankBalance:
    balance = ObservedNumber(minimum=0)

    def __init__(self, opening):
        self._listeners = []
        self.balance = opening

    def subscribe(self, callback):
        self._listeners.append(callback)

balance = BankBalance(100)
events = []
balance.subscribe(lambda name, previous, current: events.append((name, previous, current)))
balance.balance = 125
balance.balance = 125
assert events == [('balance', 100, 125)]
print('events:', events)

events: [('balance', 100, 125)]


### Step 3 — Test the failure boundary

Validation precedes mutation. This is a critical ordering decision: invalid values must not be visible to observers and must not change the model. This demonstration uses a strict nonnegative amount; real financial systems typically use `decimal.Decimal` and an explicit currency model instead of binary floats.

In [34]:
expect_error(ValueError, lambda: setattr(balance, 'balance', -1))
expect_error(ValueError, lambda: setattr(balance, 'balance', float('nan')))
expect_error(ValueError, lambda: setattr(balance, 'balance', True))
assert balance.balance == 125
assert events == [('balance', 100, 125)]
print('PASS: invalid changes are neither stored nor emitted')

PASS: invalid changes are neither stored nor emitted


### Complete solution: what happens when an observer fails?

Because notification follows commit, observers can read the new value. If a listener raises, the setter propagates the exception and **does not** undo the successful mutation. That policy must be documented for API consumers, especially if observers have their own side effects.

In [35]:
def broken_listener(name, previous, current):
    raise RuntimeError('notification failed')

balance.subscribe(broken_listener)
expect_error(RuntimeError, lambda: setattr(balance, 'balance', 150))
assert balance.balance == 150
assert events[-1] == ('balance', 125, 150)
print('PASS: post-commit event failure semantics are explicit')

PASS: post-commit event failure semantics are explicit


**Takeaway.** Validation, mutation, and event delivery are distinct phases. Descriptors can make that policy reusable, but they do not turn arbitrary callbacks into atomic transactions.

# Problem 11 — Create a writable derived unit without duplicated state

**Challenge.** Represent one temperature in Celsius and offer a synchronized Fahrenheit view with both read and write access.

**By the end, you should be able to:** use a source-of-truth data descriptor and a derived data descriptor without maintaining a stale second value.

We will move from a small experiment to a complete implementation, then check the behavior with executable assertions.

### Step 1 — Choose one authoritative representation

If both Celsius and Fahrenheit are stored separately, they can diverge. Store only Celsius and calculate Fahrenheit on demand. The Fahrenheit setter converts input to Celsius and delegates to the Celsius validator. In both directions, forbid non-finite values and Boolean masquerading as a number.

In [36]:
def finite_temperature(value):
    if type(value) not in (int, float) or not math.isfinite(value):
        raise TypeError('temperature must be a finite numeric value (not bool)')

class CelsiusField:
    def __set_name__(self, owner, name):
        self.name = name
        self.key = f'_{name}'

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if self.key not in instance.__dict__:
            raise AttributeError('Celsius not initialized')
        return instance.__dict__[self.key]

    def __set__(self, instance, value):
        finite_temperature(value)
        instance.__dict__[self.key] = value

    def __delete__(self, instance):
        raise AttributeError('temperature must remain initialized')

class FahrenheitView:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.celsius * 9 / 5 + 32

    def __set__(self, instance, value):
        finite_temperature(value)
        instance.celsius = (value - 32) * 5 / 9

    def __delete__(self, instance):
        raise AttributeError('delete the owning temperature object instead')

### Step 2 — Assemble the domain object

The constructor assigns only Celsius. The Fahrenheit view does not create an extra instance entry; every read reflects the current authoritative value. The conversion formula is intentionally visible in the descriptor, which makes its behavior easy to audit.

In [37]:
class Temperature:
    celsius = CelsiusField()
    fahrenheit = FahrenheitView()

    def __init__(self, celsius):
        self.celsius = celsius

temperature = Temperature(0)
assert temperature.fahrenheit == 32
temperature.fahrenheit = 212
assert abs(temperature.celsius - 100) < 1e-12
assert '_fahrenheit' not in temperature.__dict__
print('state:', temperature.__dict__)

state: {'_celsius': 100.0}


### Step 3 — Prevent inconsistent or impossible states

Because both public attributes are data descriptors, ordinary instance keys cannot shadow them. The Celsius validator is always used by the Fahrenheit setter, so validation logic is not duplicated. This example accepts values below absolute zero because it models mathematical unit conversion; a physical-temperature domain model could add an explicit lower bound.

In [38]:
before = temperature.celsius
for invalid in (True, 'hot', float('inf'), float('nan')):
    expect_error(TypeError, lambda invalid=invalid: setattr(temperature, 'fahrenheit', invalid))
assert temperature.celsius == before
expect_error(AttributeError, lambda: delattr(temperature, 'celsius'))
expect_error(AttributeError, lambda: delattr(temperature, 'fahrenheit'))
assert isinstance(Temperature.fahrenheit, FahrenheitView)
print('PASS: conversion, no stale cache, validation and deletion policies')

PASS: conversion, no stale cache, validation and deletion policies


**Takeaway.** Descriptors are useful for *writable computed views*. Prefer one source of truth over two synchronized copies whenever the domain allows it.

# Problem 12 — Collect inherited fields into a tiny model schema

**Challenge.** Define a record-like class whose fields register automatically, inherit predictably, and support required values and independent list defaults.

**By the end, you should be able to:** combine field descriptors with a small metaclass and avoid mutable shared defaults.

We will move from a small experiment to a complete implementation, then check the behavior with executable assertions.

### Step 1 — Define scope before writing an ORM

We are *not* creating a database, query builder, serializer for untrusted input, or production ORM. The narrow goal is a reusable schema that discovers descriptors declared on a class and those inherited from bases. It must also recognize when a subclass *replaces* an inherited field with a non-field attribute.

A descriptor's `__set_name__` runs during class creation; a metaclass can then gather fields into a class-level registry. Inherited descriptors generally do not get `__set_name__` called again.

In [39]:
_NO_DEFAULT = object()

class RecordField:
    def __init__(self, *, default=_NO_DEFAULT, default_factory=None, validator=None):
        if default is not _NO_DEFAULT and default_factory is not None:
            raise ValueError('choose default OR default_factory')
        self.default = default
        self.default_factory = default_factory
        self.validator = validator

    def __set_name__(self, owner, name):
        self.name = name
        self.key = f'_record_{name}'

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if self.key not in instance.__dict__:
            if self.default_factory is not None:
                self.__set__(instance, self.default_factory())
            elif self.default is not _NO_DEFAULT:
                self.__set__(instance, self.default)
            else:
                raise AttributeError(f'{self.name} is required')
        return instance.__dict__[self.key]

    def __set__(self, instance, value):
        if self.validator is not None and not self.validator(value):
            raise ValueError(f'invalid value for {self.name}: {value!r}')
        instance.__dict__[self.key] = value

### Step 2 — Gather fields in definition order

The metaclass first merges base registries, then processes the new class namespace in order. A newly declared `RecordField` overrides the inherited entry of that name. An ordinary attribute with a matching name removes it from the registry because it shadows the inherited field.

Multiple inheritance can introduce ambiguous field names. This simple implementation uses the base order shown in `bases`; a production framework must state and rigorously test its conflict-resolution policy.

In [40]:
class RecordMeta(type):
    def __new__(mcls, name, bases, namespace):
        fields = {}
        for base in bases:
            fields.update(getattr(base, '_fields', {}))
        for attribute, value in namespace.items():
            if isinstance(value, RecordField):
                fields[attribute] = value
            else:
                fields.pop(attribute, None)
        cls = super().__new__(mcls, name, bases, namespace)
        cls._fields = fields
        return cls

class Record(metaclass=RecordMeta):
    def __init__(self, **values):
        unexpected = values.keys() - self._fields.keys()
        if unexpected:
            raise TypeError(f'unexpected fields: {sorted(unexpected)}')
        for name, value in values.items():
            setattr(self, name, value)
        for name in self._fields:
            getattr(self, name)  # materialize defaults; reject missing required fields

    @classmethod
    def field_names(cls):
        return tuple(cls._fields)

    def to_dict(self):
        return {name: getattr(self, name) for name in self._fields}

### Step 3 — Make actual schema declarations

Use `default_factory=list` so each record gets its *own list*. Using `default=[]` would share the same list object across instances in this implementation. The field validator should reject unexpected types explicitly; here we make strings nonempty and counts genuine integers (not booleans).

In [41]:
nonempty_string = lambda value: isinstance(value, str) and bool(value.strip())
nonnegative_int = lambda value: type(value) is int and value >= 0

class BookRecord(Record):
    title = RecordField(validator=nonempty_string)
    pages = RecordField(default=0, validator=nonnegative_int)
    tags = RecordField(default_factory=list, validator=lambda value: isinstance(value, list))

class DigitalBook(BookRecord):
    format = RecordField(default='epub', validator=nonempty_string)
    pages = RecordField(default=1, validator=nonnegative_int)

book_a = BookRecord(title='Descriptors')
book_b = BookRecord(title='Methods')
book_a.tags.append('python')
assert book_a.tags == ['python']
assert book_b.tags == []
assert book_a.tags is not book_b.tags
assert DigitalBook.field_names() == ('title', 'pages', 'tags', 'format')
print('digital schema:', DigitalBook.field_names())

digital schema: ('title', 'pages', 'tags', 'format')


### Step 4 — Test inheritance, overrides, and invalid inputs

When a subclass inherits `title`, it inherits the *same descriptor object* and the same backing-key convention. This is safe because the descriptor stores per-record values on individual owners. An explicit new `pages` descriptor overrides the base field; its own `__set_name__` was called while creating the subclass.

In [42]:
digital = DigitalBook(title='E-book')
assert digital.to_dict() == {'title': 'E-book', 'pages': 1, 'tags': [], 'format': 'epub'}
assert DigitalBook.title is BookRecord.title
assert DigitalBook.pages is not BookRecord.pages
expect_error(AttributeError, lambda: BookRecord())
expect_error(TypeError, lambda: BookRecord(title='Valid', unknown=1))
expect_error(ValueError, lambda: BookRecord(title='Valid', pages=True))
expect_error(ValueError, lambda: BookRecord(title='   '))
print('PASS: required fields, overrides, default factories and validation')

PASS: required fields, overrides, default factories and validation


### Step 5 — Shadow an inherited field with an ordinary attribute

This is a surprising case: the subclass's ordinary `title` hides the base descriptor when looked up normally. The schema must not claim it is still a field. Our registry removes such names. Subclass-level shadowing should be an intentional design decision in a full framework.

In [43]:
class UntitledBook(BookRecord):
    title = 'TITLE NOT REQUIRED'

assert UntitledBook.field_names() == ('pages', 'tags')
untitled = UntitledBook()
assert untitled.title == 'TITLE NOT REQUIRED'
assert untitled.to_dict() == {'pages': 0, 'tags': []}
print('PASS: ordinary class override removes inherited field from registry')

PASS: ordinary class override removes inherited field from registry


### Complete solution: inspect what is stored

Only owner values live in each record's instance dictionary. Descriptors and the schema registry live on the class. `to_dict()` is a **shallow** view: mutable list values are returned by reference. A production serialization layer would make copying and accepted types explicit.

In [44]:
assert set(digital.__dict__) == {'_record_title', '_record_pages',
                                 '_record_tags', '_record_format'}
assert all(isinstance(field, RecordField) for field in DigitalBook._fields.values())
print('PASS: field metadata belongs to class, values to instance')

PASS: field metadata belongs to class, values to instance


**Takeaway.** Descriptors handle individual fields; a metaclass handles the *collection* and inheritance policy. Separate these responsibilities and specify how overrides, defaults, and input validation behave.

# Problem 13 — Inspect attributes without accidentally running their descriptors

**Challenge.** Write debugging code that identifies a descriptor without triggering a costly or state-changing `__get__`.

**By the end, you should be able to:** use `inspect.getattr_static` and understand why `getattr`, `hasattr`, and even direct class access can execute code.

We will move from a small experiment to a complete implementation, then check the behavior with executable assertions.

### Step 1 — Set up an intentionally side-effectful descriptor

In application code, `__get__` may issue a database request, increment a counter, or raise an exception. Merely asking whether an attribute exists via `hasattr(obj, name)` runs ordinary lookup: it calls the descriptor and returns `False` only for `AttributeError`. It does **not** guarantee side-effect-free discovery.

In [45]:
import inspect

class Metered:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        instance.read_count += 1
        return f'value-{instance.read_count}'

class ObservedService:
    metric = Metered()

    def __init__(self):
        self.read_count = 0

service = ObservedService()
assert service.metric == 'value-1'
assert hasattr(service, 'metric') is True
assert service.read_count == 2
print('normal access and hasattr caused', service.read_count, 'reads')

normal access and hasattr caused 2 reads


### Step 2 — Reach the *raw* class attribute

`inspect.getattr_static` is designed for static attribute retrieval; it does not invoke normal dynamic descriptor lookup. This is useful for inspection tools. However, its returned value is not necessarily what ordinary `getattr` would return, and it has documented corner cases involving dynamically resolved or shadowed attributes.

In [46]:
reads_before = service.read_count
raw = inspect.getattr_static(service, 'metric')
assert isinstance(raw, Metered)
assert raw is ObservedService.__dict__['metric']
assert service.read_count == reads_before
assert vars(service) == {'read_count': reads_before}
print('PASS: static inspection did not execute __get__')

PASS: static inspection did not execute __get__


### Step 3 — Demonstrate why `hasattr` is not an error probe

A descriptor may raise `AttributeError` to say a value is not initialized, and then `hasattr` will report `False`. Any other exception propagates. For debugging or schema inspection, examine raw class attributes; for ordinary business behavior, use normal access and handle expected exceptions deliberately.

In [47]:
class MissingOnAccess:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        raise AttributeError('not available right now')

class Unavailable:
    thing = MissingOnAccess()

unavailable = Unavailable()
assert not hasattr(unavailable, 'thing')
assert isinstance(inspect.getattr_static(unavailable, 'thing'), MissingOnAccess)
print('PASS: presence of a descriptor differs from successful normal lookup')

PASS: presence of a descriptor differs from successful normal lookup


**Takeaway.** Introspection is itself an API decision. Use `inspect.getattr_static` when you need the raw definition; use `getattr` when you actually intend to execute descriptor semantics.

# Problem 14 — Preserve a cross-field invariant with an atomic update method

**Challenge.** Represent a date window where `start <= end` and ensure failed updates never leave a half-updated object.

**By the end, you should be able to:** separate field-level descriptor checks from object-level invariants, and validate both new values before committing either one.

We will move from a small experiment to a complete implementation, then check the behavior with executable assertions.

### Step 1 — Locate the real abstraction boundary

A field descriptor can validate a single date, but it cannot reliably enforce the range invariant when `start` and `end` are set independently in an arbitrary order. For instance, changing `start` first might temporarily conflict with the old `end` even though the intended *pair* is valid.

The correct public API is `set_range(start, end)`: validate the entire proposed state, then commit both values. Read-only descriptors prevent casual independent assignment.

In [48]:
from datetime import date

class ReadOnlyDate:
    def __set_name__(self, owner, name):
        self.name = name
        self.key = f'_window_{name}'

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.key]
        except KeyError:
            raise AttributeError(f'{self.name} is not initialized') from None

    def __set__(self, instance, value):
        raise AttributeError('use set_range(start, end) to update both dates')

    def __delete__(self, instance):
        raise AttributeError('date window fields cannot be deleted')

class DateWindow:
    start = ReadOnlyDate()
    end = ReadOnlyDate()

    def __init__(self, start, end):
        self.set_range(start, end)

    def set_range(self, start, end):
        if type(start) is not date or type(end) is not date:
            raise TypeError('both endpoints must be date objects, not datetime')
        if start > end:
            raise ValueError('start must not follow end')
        # All validations have passed; only now do we publish the new pair.
        self.__dict__.update({type(self).start.key: start,
                              type(self).end.key: end})

### Step 2 — Explain the implementation boundary

The method deliberately writes to private backing keys rather than assigning to the public descriptor properties, because their `__set__` rejects external field-by-field updates. This centralizes the valid mutation path. It is not an isolation primitive against untrusted Python code that directly edits `__dict__`.

Our validators use `type(value) is date`, which excludes `datetime`: a subclass of `date` that would introduce time-of-day semantics into a date-only window.

In [49]:
window = DateWindow(date(2030, 1, 1), date(2030, 1, 31))
window.set_range(date(2030, 2, 1), date(2030, 2, 28))
assert window.start == date(2030, 2, 1)
assert window.end == date(2030, 2, 28)
expect_error(ValueError, lambda: window.set_range(date(2030, 4, 1), date(2030, 3, 1)))
assert (window.start, window.end) == (date(2030, 2, 1), date(2030, 2, 28))
print('PASS: invalid range leaves both previous values unchanged')

PASS: invalid range leaves both previous values unchanged


### Step 3 — Block accidental partial updates and wrong types

A failed check must occur before either backing key is mutated. Tests should verify *both* fields afterward. This is transaction-like behavior within a single-threaded object method, not a thread-safe database transaction; concurrency requires a lock or stronger coordination.

In [50]:
expect_error(AttributeError, lambda: setattr(window, 'start', date(2030, 1, 1)))
expect_error(AttributeError, lambda: delattr(window, 'end'))
expect_error(TypeError, lambda: window.set_range(datetime(2030, 1, 1), date(2030, 2, 1)))
assert (window.start, window.end) == (date(2030, 2, 1), date(2030, 2, 28))
assert isinstance(DateWindow.start, ReadOnlyDate)
print('PASS: one public update route enforces the cross-field invariant')

PASS: one public update route enforces the cross-field invariant


**Takeaway.** Not every invariant belongs in one field descriptor. Use descriptors for reusable local rules and an owner method for constraints spanning multiple fields.

# Problem 15 — Combine a public descriptor with private `__slots__` storage

**Challenge.** Validate a public attribute on a compact object that intentionally has no `__dict__`.

**By the end, you should be able to:** avoid a slot/descriptor name collision and use `object.__getattribute__` or direct private-slot access safely.

We will move from a small experiment to a complete implementation, then check the behavior with executable assertions.

### Step 1 — Understand the name-collision trap

A slot declaration creates a slot descriptor on the class. Defining a second class attribute with the same name as that slot is invalid. The solution is to use a **different** private slot for storage, such as `_score`, and a public validating descriptor named `score`.

This storage design needs neither an instance dictionary nor weak-key mapping. It is appropriate when each owner class can declare the required backing slots.

In [51]:
class SlottedScore:
    def __set_name__(self, owner, name):
        self.name = name
        self.slot = f'_{name}'

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return object.__getattribute__(instance, self.slot)
        except AttributeError:
            raise AttributeError(f'{self.name} not initialized') from None

    def __set__(self, instance, value):
        if type(value) is not int or not 0 <= value <= 100:
            raise ValueError(f'{self.name} must be an integer in [0, 100]')
        object.__setattr__(instance, self.slot, value)

    def __delete__(self, instance):
        raise AttributeError('score cannot be deleted')

class ExamResult:
    __slots__ = ('_score',)
    score = SlottedScore()

    def __init__(self, score):
        self.score = score

### Step 2 — Exercise normal access, validation, and storage

The `score` descriptor is a class attribute. `_score` is a distinct slot created by Python. `object.__setattr__` respects the private slot descriptor and avoids accidentally calling the public `score` descriptor again. We do not need a custom `__getattribute__` implementation.

In [52]:
exam = ExamResult(82)
assert not hasattr(exam, '__dict__')
assert exam.score == 82
assert object.__getattribute__(exam, '_score') == 82
exam.score = 95
assert exam.score == 95
expect_error(ValueError, lambda: setattr(exam, 'score', True))
expect_error(ValueError, lambda: setattr(exam, 'score', 101))
assert exam.score == 95
print('PASS: validated public descriptor works with private slot storage')

PASS: validated public descriptor works with private slot storage


### Step 3 — Confirm class-level introspection and the missing-value path

Class access returns the descriptor. When an object is allocated with `__new__` without running its constructor, the slot has no value and the getter translates that absence into a meaningful `AttributeError`. In a regular program, prefer proper construction; `__new__` is used here solely to test the boundary.

In [53]:
assert ExamResult.score is ExamResult.__dict__['score']
raw_exam = ExamResult.__new__(ExamResult)
expect_error(AttributeError, lambda: raw_exam.score)
expect_error(AttributeError, lambda: delattr(exam, 'score'))
print('PASS: introspection and uninitialized slot behavior')

PASS: introspection and uninitialized slot behavior


**Takeaway.** There are at least three useful per-instance storage designs: an owner dictionary, an external weak-key map, and an explicit private slot. Choose based on the owner object's capabilities and lifecycle requirements.

---
# Synthesis: choose the correct descriptor design

When designing a new descriptor, ask these questions *before* coding:

1. **Is it data or non-data?** Do you want the instance dictionary to override it? A `__delete__` method alone changes the answer.
2. **Where does mutable state live?** Usually on the owner instance; for slotted owners, consider a backing slot or weak-key storage.
3. **What does class access return?** Returning `self` supports introspection and avoids surprising class-level computation.
4. **How are names discovered?** `__set_name__` is useful, but inherited descriptors do not get a second call automatically.
5. **How is invalid input handled?** Validate before mutation; reject `bool` when numeric business values require it.
6. **What are the cache rules?** Specify invalidation, expiry, failure handling, concurrency, and clock selection.
7. **Are there cross-field invariants?** Keep those in an owner-level update operation, not a single-field setter.
8. **Is this an access policy or actual security?** Ordinary descriptors and `__dict__` conventions are not security barriers against arbitrary Python code.

## Further practice — solve without looking back

These optional prompts are deliberately left as exercises; **all 15 main problems above include fully worked code and assertions**.

- Add a `coerce` function to `BoundedNumber`, but avoid silently interpreting `True` as `1`.
- Extend `LazyValue` with a read-only policy and explain why adding `__set__` forces you to redesign caching.
- Add an explicit `reset()` API to `RandomChoice` without putting generator state on the descriptor.
- Add a `subscribe_once()` operation to `BankBalance` and define how listener errors should behave.
- Extend `RecordMeta` with a documented multiple-inheritance conflict check.
- Add a lock to `TTLValue` and explain how to prevent duplicate refreshes without blocking unrelated owner instances.

For every extension, write tests for the normal behavior, failure path, independent instances, and class-level access.

## Final notebook self-check

Running the following cell successfully means that the notebook's examples have reached the expected state. It is not a substitute for dedicated unit tests in a production package, but it gives immediate feedback when studying interactively.

In [54]:
assert Event.time is Event.__dict__['time']
assert p.data == 'data result'
assert first.tax_rate == 7.5
assert job_a.job_id == 'A-001'
assert len(sequence_a) == 8
assert report.calculations == 3
assert client_a.loads == 3
assert y.temperature == 23
assert isinstance(AdminAction.run, GuardedMethod)
assert balance.balance == 150
assert abs(temperature.celsius - 100) < 1e-12
assert digital.pages == 1
assert service.read_count == 2
assert window.start <= window.end
assert exam.score == 95
print('ALL 15 TUTORIAL PROBLEMS: FINAL STATE CHECK PASSED')

ALL 15 TUTORIAL PROBLEMS: FINAL STATE CHECK PASSED
